In [17]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

In [18]:
files = [
    "data/CRMLSSold202506.csv", "data/CRMLSSold202507.csv", "data/CRMLSSold202508.csv",
    "data/CRMLSSold202509.csv", "data/CRMLSSold202510.csv", "data/CRMLSSold202511.csv",
    "data/CRMLSSold202512.csv", "data/CRMLSSold202601.csv", "data/CRMLSSold202602.csv",
    "data/CRMLSSold202603.csv", "data/CRMLSSold202604.csv", "data/CRMLSSold202605.csv"
]

valid_files = []
for f in files:
    if os.path.exists(f):
        valid_files.append(f)
    elif os.path.exists(os.path.join("..", f)):
        valid_files.append(os.path.join("..", f))

df = pd.concat([pd.read_csv(f, low_memory=False) for f in valid_files], ignore_index=True)

# Filters from week 2
df_restricted = df[(df["PropertyType"] == "Residential") & (df["PropertySubType"] == "SingleFamilyResidence")].copy()

In [19]:
df_restricted = df_restricted.dropna(subset=["ClosePrice", "LivingArea"])

# Fill missing values with median values
df_restricted["BedroomsTotal"] = df_restricted["BedroomsTotal"].fillna(df_restricted["BedroomsTotal"].median())
df_restricted["BathroomsTotalInteger"] = df_restricted["BathroomsTotalInteger"].fillna(df_restricted["BathroomsTotalInteger"].median())
df_restricted["LotSizeSquareFeet"] = df_restricted["LotSizeSquareFeet"].fillna(df_restricted["LotSizeSquareFeet"].median())

In [20]:
features_to_scale = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet"]

scaler = StandardScaler()
df_restricted[[f"{col}_scaled" for col in features_to_scale]] = scaler.fit_transform(df_restricted[features_to_scale])

In [21]:
df_restricted["CloseDate"] = pd.to_datetime(df_restricted["CloseDate"])

# May 2026 test set
test_df = df_restricted[(df_restricted["CloseDate"].dt.year == 2026) & (df_restricted["CloseDate"].dt.month == 5)].copy()

X = 6

test_start = pd.Timestamp("2026-05-01")
train_start = test_start - pd.DateOffset(months=X)
train_df = df_restricted[(df_restricted["CloseDate"] >= train_start) & (df_restricted["CloseDate"] < test_start)].copy()

print(f"Training set size (X={X} months): {train_df.shape}")
print(f"Test set size (May 2026): {test_df.shape}")

Training set size (X=6 months): (59415, 82)
Test set size (May 2026): (12017, 82)


In [22]:
# Export to csv
output_path = "data/cleaned_sales_data.csv"

if not os.path.exists("data") and os.path.exists("../data"):
    output_path = "../data/cleaned_sales_data.csv"

df_restricted.to_csv(output_path, index=False)
print(f"Preprocessed dataset successfully saved to: {output_path}")

Preprocessed dataset successfully saved to: ../data/cleaned_sales_data.csv
